# DocLayout-YOLO: Advanced Stages Implementation

This notebook implements the multi-stage pipeline for robust template matching:

1. **Stage 1:** Geometry Pixel Similarity Detection (SSIM + ORB)
2. **Stage 2:** Embedding Model Comparison & Swap (all-MiniLM vs BAAI/bge-m3)
3. **Stage 3:** Two-Stage Retrieval (Geometry + Text Reranking)
4. **Stage 4:** OCR Integration & Field Extraction
5. **Stage 5:** Deduplication with Hash-Based IDs

Each stage builds on the previous, showing concrete results and performance comparisons.

## Install Additional Dependencies

In [2]:
import sys, subprocess

# Dependencies for real document pipeline + advanced stages
packages = [
    "torch",
    "torchvision",
    "ultralytics",
    "doclayout-yolo",
    "huggingface_hub",
    "chromadb",
    "pypdfium2",
    "Pillow",
    "numpy",
    "matplotlib",
    "opencv-python",
    "scikit-image",
    "sentence-transformers",
    "paddleocr",
]

for pkg in packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"[OK] {pkg}")
    except Exception as e:
        print(f"[WARN] {pkg}: {e}")

print("\nDependency installation complete.")

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
✓ opencv-python
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
✓ scikit-image
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
✓ sentence-transformers
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
✓ paddleocr

Dependency installation complete.


## Imports

In [3]:
import os
from pathlib import Path
import hashlib
from typing import Dict, List, Tuple
from collections import Counter

import numpy as np
from PIL import Image
import cv2
from skimage.metrics import structural_similarity as ssim
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pypdfium2 as pdfium

from huggingface_hub import hf_hub_download
from doclayout_yolo import YOLOv10
from sentence_transformers import SentenceTransformer
from paddleocr import PaddleOCR

print("[OK] Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

/home/dellpromax/Documents/shivam/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports successful
PyTorch version: 2.13.0+cu130
CUDA available: True


---

# STAGE 1: Geometry Pixel Similarity Detection

Compare two document pages pixel-by-pixel using:
- **SSIM** (Structural Similarity Index): Fast, no learning
- **ORB** (Oriented FAST and Rotated BRIEF): Keypoint matching, robust to distortions

In [14]:
# Shared configuration for real data execution
import pypdfium2 as pdfium
from collections import Counter
from huggingface_hub import hf_hub_download
from doclayout_yolo import YOLOv10

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMGSZ = 1024
CONF_THRESHOLD = 0.2
TOP_N_RESULTS = 3

ROOT_DIR = Path.cwd()
DATA_ROOT = ROOT_DIR / "yolostructuredetection" / "test"
if not DATA_ROOT.exists():
    DATA_ROOT = ROOT_DIR
TEMPLATE_DIR = DATA_ROOT / "templates"
TEST_DIR = DATA_ROOT / "test"

CLASS_NAMES = {
    0: "title",
    1: "plain_text",
    2: "abandon",
    3: "figure",
    4: "figure_caption",
    5: "table",
    6: "table_caption",
    7: "table_footnote",
    8: "isolate_formula",
    9: "formula_caption",
}

_MEANINGFUL = {
    "title", "plain_text", "figure", "figure_caption",
    "table", "table_caption", "table_footnote",
    "isolate_formula", "formula_caption",
}

def pdf_to_images(pdf_path: str | Path, scale: float = 2.0) -> list[Image.Image]:
    doc = pdfium.PdfDocument(str(pdf_path))
    return [page.render(scale=scale).to_pil().convert("RGB") for page in doc]

def load_document_pages(file_path: str | Path) -> list[Image.Image]:
    path = Path(file_path)
    ext = path.suffix.lower()
    if ext == ".pdf":
        return pdf_to_images(path)
    if ext in (".png", ".jpg", ".jpeg", ".tiff", ".bmp", ".gif", ".webp"):
        return [Image.open(path).convert("RGB")]
    return []

def _spatial_zone(cx: float, cy: float) -> str:
    col = "left" if cx < 0.35 else ("centre" if cx < 0.65 else "right")
    row = "top" if cy < 0.33 else ("middle" if cy < 0.67 else "bottom")
    return f"{row}-{col}"

def build_layout_descriptor(detections: list[dict]) -> str:
    if not detections:
        return "empty document layout: no structural elements detected"

    counts = Counter(d["class_name"] for d in detections)
    total = len(detections)
    count_str = ", ".join(f"{v} {k.replace('_', ' ')}" for k, v in sorted(counts.items()))

    grid: dict[str, set[str]] = {}
    for d in detections:
        zone = _spatial_zone(d["cx_norm"], d["cy_norm"])
        grid.setdefault(zone, set()).add(d["class_name"].replace("_", " "))
    grid_str = "; ".join(
        f"{zone}: [{', '.join(sorted(items))}]"
        for zone, items in sorted(grid.items())
    )

    meaningful_counts = {k: v for k, v in counts.items() if k in _MEANINGFUL}
    dominant = max(meaningful_counts, key=meaningful_counts.get) if meaningful_counts else "none"

    sequence = [
        f"{d['class_name'].replace('_', ' ')} at {_spatial_zone(d['cx_norm'], d['cy_norm'])}"
        for d in detections if d["class_name"] in _MEANINGFUL
    ]

    area_by_class: dict[str, float] = {}
    for d in detections:
        if d["class_name"] in _MEANINGFUL:
            area_by_class[d["class_name"]] = area_by_class.get(d["class_name"], 0.0) + d["area_norm"]
    coverage_str = ", ".join(
        f"{k.replace('_', ' ')} {v*100:.1f}%"
        for k, v in sorted(area_by_class.items(), key=lambda x: -x[1])
    )

    return (
        f"Document structural layout with {total} detected regions. "
        f"Element counts: {count_str}. "
        f"Dominant element: {dominant.replace('_', ' ')}. "
        f"Spatial grid (zone: [types]): {grid_str}. "
        f"Page coverage: {coverage_str}. "
        f"Reading-order sequence: {'; '.join(sequence)}."
    )

def detect_layout(image: Image.Image, model) -> list[dict]:
    img_w, img_h = image.size
    results = model.predict(
        image, imgsz=IMGSZ, conf=CONF_THRESHOLD, device=DEVICE, verbose=False,
    )

    detections: list[dict] = []
    if results and results[0].boxes is not None:
        boxes = results[0].boxes
        for i in range(len(boxes)):
            x1, y1, x2, y2 = boxes.xyxy[i].cpu().numpy().tolist()
            cls = int(boxes.cls[i].item())
            conf = float(boxes.conf[i].item())
            cx = (x1 + x2) / 2 / img_w
            cy = (y1 + y2) / 2 / img_h
            w = (x2 - x1) / img_w
            h = (y2 - y1) / img_h
            detections.append({
                "class_id": cls,
                "class_name": CLASS_NAMES.get(cls, f"class_{cls}"),
                "conf": round(conf, 3),
                "cx_norm": round(cx, 4),
                "cy_norm": round(cy, 4),
                "w_norm": round(w, 4),
                "h_norm": round(h, 4),
                "area_norm": round(w * h, 5),
            })
    detections.sort(key=lambda d: d["cy_norm"])
    return detections

print("Loading DocLayout-YOLO checkpoint...")
model_path = hf_hub_download(
    repo_id="juliozhao/DocLayout-YOLO-DocStructBench",
    filename="doclayout_yolo_docstructbench_imgsz1024.pt",
)
layout_model = YOLOv10(model_path)
print(f"[OK] Model loaded: {model_path}")
print(f"[OK] Data root    : {DATA_ROOT}")
print(f"[OK] Templates    : {TEMPLATE_DIR}")
print(f"[OK] Test docs    : {TEST_DIR}")

Loading DocLayout-YOLO checkpoint...
[OK] Model loaded: /home/dellpromax/.cache/huggingface/hub/models--juliozhao--DocLayout-YOLO-DocStructBench/snapshots/8c3299a30b8ff29a1503c4431b035b93220f7b11/doclayout_yolo_docstructbench_imgsz1024.pt
[OK] Data root    : /home/dellpromax/Documents/shivam/yolostructuredetection/test
[OK] Templates    : /home/dellpromax/Documents/shivam/yolostructuredetection/test/templates
[OK] Test docs    : /home/dellpromax/Documents/shivam/yolostructuredetection/test/test


### Stage 1 Run on Original Documents

Compare real files from `templates/` and `test/` using SSIM + ORB and report the best geometric match per input file.

In [12]:
# Stage 1 on real documents: pixel-level geometry similarity between test docs and template pages
def compare_layouts_ssim(img1: Image.Image, img2: Image.Image) -> float:
    size = (640, 900)
    arr1 = np.array(img1.resize(size, Image.LANCZOS).convert("L"), dtype=np.float32)
    arr2 = np.array(img2.resize(size, Image.LANCZOS).convert("L"), dtype=np.float32)
    score, _ = ssim(arr1, arr2, full=True, data_range=255)
    return float(score)

def compare_layouts_orb(img1: Image.Image, img2: Image.Image) -> float:
    img1_cv = cv2.cvtColor(np.array(img1), cv2.COLOR_RGB2GRAY)
    img2_cv = cv2.cvtColor(np.array(img2), cv2.COLOR_RGB2GRAY)
    size = (640, 900)
    img1_cv = cv2.resize(img1_cv, size)
    img2_cv = cv2.resize(img2_cv, size)

    orb = cv2.ORB_create(nfeatures=1000)
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

    kp1, des1 = orb.detectAndCompute(img1_cv, None)
    kp2, des2 = orb.detectAndCompute(img2_cv, None)
    if des1 is None or des2 is None or len(kp1) == 0 or len(kp2) == 0:
        return 0.0

    matches = bf.match(des1, des2)
    max_matches = max(len(kp1), len(kp2))
    return float(len(matches) / max_matches) if max_matches > 0 else 0.0

def list_supported_files(folder: Path) -> list[Path]:
    allowed = {".pdf", ".png", ".jpg", ".jpeg", ".tiff", ".bmp", ".gif", ".webp"}
    if not folder.exists():
        return []
    return [p for p in sorted(folder.iterdir()) if p.is_file() and p.suffix.lower() in allowed]

template_files = list_supported_files(TEMPLATE_DIR)
test_files = list_supported_files(TEST_DIR)

if not template_files or not test_files:
    raise RuntimeError("Template/test files not found. Check TEMPLATE_DIR and TEST_DIR.")

print("\n" + "=" * 72)
print("STAGE 1 (REAL DATA): PIXEL GEOMETRY SIMILARITY")
print("=" * 72)

results_stage1 = []
for test_file in test_files:
    test_pages = load_document_pages(test_file)
    if not test_pages:
        continue
    test_img = test_pages[0]

    best = None
    for tpl_file in template_files:
        tpl_pages = load_document_pages(tpl_file)
        if not tpl_pages:
            continue
        tpl_img = tpl_pages[0]

        ssim_score = compare_layouts_ssim(test_img, tpl_img)
        orb_score = compare_layouts_orb(test_img, tpl_img)
        hybrid = 0.5 * ssim_score + 0.5 * orb_score

        row = {
            "query": test_file.name,
            "template": tpl_file.name,
            "ssim": round(ssim_score, 4),
            "orb": round(orb_score, 4),
            "hybrid_score": round(hybrid, 4),
        }
        if best is None or row["hybrid_score"] > best["hybrid_score"]:
            best = row

    if best:
        results_stage1.append(best)
        print(f"\nQuery: {best['query']}")
        print(f"  Best template : {best['template']}")
        print(f"  SSIM          : {best['ssim']:.4f}")
        print(f"  ORB           : {best['orb']:.4f}")
        print(f"  Hybrid        : {best['hybrid_score']:.4f}")

print("\n" + "=" * 72)


STAGE 1 (REAL DATA): PIXEL GEOMETRY SIMILARITY

Query: certificat of origine..pdf
  Best template : Egypt- CoO Templates.pdf
  SSIM          : 0.5114
  ORB           : 0.2840
  Hybrid        : 0.3977

Query: certificat of origine.pdf
  Best template : CoO WCO (kyoto).png
  SSIM          : 0.5718
  ORB           : 0.2370
  Hybrid        : 0.4044



---

# STAGE 2: Embedding Model Comparison (Real Descriptors)

Compare embedding quality on descriptors generated from your actual template pages:
- **all-MiniLM-L6-v2** (current default baseline)
- **BAAI/bge-m3** (strong text retrieval, 568M params)

In [15]:
# Stage 2 on real documents: embedding comparison using actual layout descriptors
print("Loading embedding models (this may take a minute)...\n")

models = {
    "all-MiniLM-L6-v2": SentenceTransformer("all-MiniLM-L6-v2"),
    "BAAI/bge-m3": SentenceTransformer("BAAI/bge-m3"),
}
print("[OK] Models loaded\n")

# Build descriptor dataset from real template pages
template_records = []
for tpl_file in template_files:
    pages = load_document_pages(tpl_file)
    for page_idx, img in enumerate(pages, start=1):
        detections = detect_layout(img, layout_model)
        descriptor = build_layout_descriptor(detections)
        template_records.append({
            "id": f"{tpl_file.name}::p{page_idx}",
            "template_file": tpl_file.name,
            "page": page_idx,
            "image": img,
            "detections": detections,
            "descriptor": descriptor,
        })

if not template_records:
    raise RuntimeError("No template records were generated from real data.")

test_descriptors = [r["descriptor"] for r in template_records[: min(8, len(template_records))]]
print(f"Using {len(test_descriptors)} real descriptors for model comparison.\n")

embedding_results = {}
from sklearn.metrics.pairwise import cosine_similarity

for model_name, model in models.items():
    print(f"  {model_name}...")
    embeddings = model.encode(test_descriptors)
    sim_matrix = cosine_similarity(embeddings)
    embedding_results[model_name] = {
        "embeddings": embeddings,
        "similarity_matrix": sim_matrix,
        "model_size": model.get_sentence_embedding_dimension(),
        "max_seq_length": model.max_seq_length,
    }

print("\n" + "=" * 72)
print("STAGE 2 (REAL DATA): EMBEDDING MODEL COMPARISON")
print("=" * 72)
for model_name, data in embedding_results.items():
    print(f"\n{model_name}:")
    print(f"  Embedding dimension  : {data['model_size']}")
    print(f"  Max sequence length  : {data['max_seq_length']}")
    print("  Avg off-diagonal similarity (real descriptors):")
    sim = data["similarity_matrix"]
    if sim.shape[0] > 1:
        mask = ~np.eye(sim.shape[0], dtype=bool)
        print(f"    {sim[mask].mean():.4f}")
    else:
        print("    n/a (single descriptor)")
print("\n" + "=" * 72)

Loading embedding models (this may take a minute)...



Loading weights: 100%|██████████| 391/391 [00:00<00:00, 18755.41it/s]


[OK] Models loaded

Using 8 real descriptors for model comparison.

  all-MiniLM-L6-v2...
  BAAI/bge-m3...

STAGE 2 (REAL DATA): EMBEDDING MODEL COMPARISON

all-MiniLM-L6-v2:
  Embedding dimension  : 384
  Max sequence length  : 256
  Avg off-diagonal similarity (real descriptors):
    0.9363

BAAI/bge-m3:
  Embedding dimension  : 1024
  Max sequence length  : 8192
  Avg off-diagonal similarity (real descriptors):
    0.9736



/tmp/ipykernel_103572/3288539674.py:42: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  "model_size": model.get_sentence_embedding_dimension(),
/tmp/ipykernel_103572/3288539674.py:42: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  "model_size": model.get_sentence_embedding_dimension(),


---

# STAGE 3: Two-Stage Retrieval Pipeline

Combine geometry + text semantics for robust matching:
1. **Stage A (Coarse):** Retrieve by geometry descriptor similarity
2. **Stage B (Fine):** Rerank by text field + table semantic similarity

In [18]:
class TwoStageRetriever:
    """Two-stage template matching: descriptor coarse match + OCR rerank."""

    def __init__(self, embedding_model_name: str = "BAAI/bge-m3"):
        self.embedding_model = SentenceTransformer(embedding_model_name)
        self.model_name = embedding_model_name

    def descriptor_similarity(self, q_desc: str, t_desc: str) -> float:
        emb = self.embedding_model.encode([q_desc, t_desc])
        from sklearn.metrics.pairwise import cosine_similarity
        return float(cosine_similarity([emb[0]], [emb[1]])[0][0])

    def text_similarity(self, q_text: str, t_text: str) -> float:
        if not q_text.strip() or not t_text.strip():
            return 0.0
        emb = self.embedding_model.encode([q_text, t_text])
        from sklearn.metrics.pairwise import cosine_similarity
        return float(cosine_similarity([emb[0]], [emb[1]])[0][0])

    def retrieve_templates(
        self,
        query_desc: str,
        query_text: str,
        templates_db: List[Dict],
        top_k_stage1: int = 5,
        alpha: float = 0.6,
        beta: float = 0.4,
    ) -> List[Dict]:
        stage1 = []
        for template in templates_db:
            g = self.descriptor_similarity(query_desc, template["descriptor"])
            stage1.append({**template, "geometry_sim": g})
        stage1.sort(key=lambda x: x["geometry_sim"], reverse=True)
        stage1 = stage1[: min(top_k_stage1, len(stage1))]

        reranked = []
        for template in stage1:
            t = self.text_similarity(query_text, template.get("ocr_text", ""))
            score = alpha * template["geometry_sim"] + beta * t
            reranked.append({
                "template_id": template["id"],
                "template_name": template["template_file"],
                "page": template["page"],
                "geometry_sim": round(template["geometry_sim"], 4),
                "text_sim": round(t, 4),
                "hybrid_score": round(score, 4),
            })
        reranked.sort(key=lambda x: x["hybrid_score"], reverse=True)
        return reranked

In [24]:
print("Initializing two-stage retriever on real documents...")
retriever = TwoStageRetriever(embedding_model_name="BAAI/bge-m3")

def make_ocr_engine() -> PaddleOCR | None:
    """Create PaddleOCR instance; return None if runtime deps are missing."""
    try:
        try:
            device = "gpu" if torch.cuda.is_available() else "cpu"
            return PaddleOCR(use_textline_orientation=True, lang="en", device=device)
        except TypeError:
            return PaddleOCR(use_angle_cls=True, lang="en")
        except ValueError:
            return PaddleOCR(use_angle_cls=True, lang="en")
    except Exception as e:
        print(f"[WARN] OCR engine unavailable, text rerank disabled: {e}")
        return None

ocr_engine = make_ocr_engine()

def ocr_page_text(image: Image.Image, max_chars: int = 1200) -> str:
    if ocr_engine is None:
        return ""
    try:
        result = ocr_engine.ocr(np.array(image), cls=True)
    except Exception as e:
        print(f"[WARN] OCR failed on page: {e}")
        return ""

    chunks = []
    for line in result:
        if not line:
            continue
        for item in line:
            txt = item[1][0] if len(item) > 1 else ""
            if txt:
                chunks.append(txt)
    joined = " ".join(chunks)
    return joined[:max_chars]

for rec in template_records:
    if "ocr_text" not in rec:
        rec["ocr_text"] = ocr_page_text(rec["image"])

print("\n" + "=" * 72)
print("STAGE 3 (REAL DATA): TWO-STAGE RETRIEVAL RESULTS")
print("=" * 72)

matches = []
for test_file in test_files:
    pages = load_document_pages(test_file)
    if not pages:
        continue
    q_img = pages[0]
    q_det = detect_layout(q_img, layout_model)
    q_desc = build_layout_descriptor(q_det)
    q_text = ocr_page_text(q_img)

    ranked = retriever.retrieve_templates(
        query_desc=q_desc,
        query_text=q_text,
        templates_db=template_records,
        top_k_stage1=5,
        alpha=0.6,
        beta=0.4,
    )
    if ranked:
        best = ranked[0]
        matches.append({"query": test_file.name, **best})
        print(f"\nQuery: {test_file.name}")
        print(f"  Best template : {best['template_name']} (page {best['page']})")
        print(f"  Geometry sim  : {best['geometry_sim']:.4f}")
        print(f"  Text sim      : {best['text_sim']:.4f}")
        print(f"  Hybrid score  : {best['hybrid_score']:.4f}")

print("\n" + "=" * 72)

Initializing two-stage retriever on real documents...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 12305.09it/s]
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/dellpromax/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.


[WARN] OCR engine unavailable, text rerank disabled: Engine 'paddle_static' is unavailable because dependency 'paddlepaddle' is not installed.

STAGE 3 (REAL DATA): TWO-STAGE RETRIEVAL RESULTS

Query: certificat of origine..pdf
  Best template : Egypt- CoO Templates.pdf (page 1)
  Geometry sim  : 0.9993
  Text sim      : 0.0000
  Hybrid score  : 0.5996

Query: certificat of origine.pdf
  Best template : Egypt- CoO Templates.pdf (page 9)
  Geometry sim  : 0.9693
  Text sim      : 0.0000
  Hybrid score  : 0.5816



---

# STAGE 4: OCR Integration & Field Extraction (Original Files)

Extract text fields from your actual documents in `templates/` and `test/`.
If OCR runtime dependencies are missing, the stage now degrades gracefully and continues.

In [20]:
class OCRFieldExtractor:
    """Extract OCR text and basic key-value fields from real documents."""

    def __init__(self):
        print("Initializing PaddleOCR...")
        self.ocr = None
        try:
            try:
                device = "gpu" if torch.cuda.is_available() else "cpu"
                self.ocr = PaddleOCR(use_textline_orientation=True, lang="en", device=device)
            except TypeError:
                self.ocr = PaddleOCR(use_angle_cls=True, lang="en")
            except ValueError:
                self.ocr = PaddleOCR(use_angle_cls=True, lang="en")
            print("[OK] PaddleOCR ready\n")
        except Exception as e:
            print(f"[WARN] PaddleOCR unavailable, OCR stage will run with empty text: {e}\n")

    def extract_text_from_image(self, image: Image.Image, confidence_threshold: float = 0.3) -> str:
        if self.ocr is None:
            return ""
        try:
            result = self.ocr.ocr(np.array(image), cls=True)
        except Exception as e:
            print(f"[WARN] OCR failed for image: {e}")
            return ""

        texts = []
        for line in result:
            if not line:
                continue
            for item in line:
                txt = item[1][0] if len(item) > 1 else ""
                conf = float(item[1][1]) if len(item) > 1 else 0.0
                if txt and conf >= confidence_threshold:
                    texts.append(txt)
        return "\n".join(texts)

    def parse_key_value_fields(self, full_text: str) -> Dict[str, str]:
        fields: Dict[str, str] = {}
        for line in full_text.splitlines():
            l = line.strip()
            if not l:
                continue
            if ":" in l:
                k, v = l.split(":", 1)
                fields[k.strip().lower()] = v.strip()
            elif "=" in l:
                k, v = l.split("=", 1)
                fields[k.strip().lower()] = v.strip()
        return fields

print("[OK] OCRFieldExtractor class defined")

[OK] OCRFieldExtractor class defined


In [28]:
print("\n" + "=" * 72)
print("STAGE 4 (REAL DATA): OCR & FIELD EXTRACTION")
print("=" * 72)

extractor = None
try:
    extractor = OCRFieldExtractor()
except Exception as e:
    print(f"[WARN] OCR extractor init failed, continuing without OCR text: {e}")

real_ocr_samples = []
for p in (template_files[:2] + test_files[:2]):
    pages = load_document_pages(p)
    if not pages:
        continue
    page = pages[0]

    if extractor is None:
        text = ""
        fields = {}
    else:
        text = extractor.extract_text_from_image(page)
        fields = extractor.parse_key_value_fields(text)

    real_ocr_samples.append({
        "file": p.name,
        "text": text,
        "fields": fields,
    })

    preview = " ".join(text.split())[:220]
    print(f"\nFile: {p.name}")
    print(f"  OCR text preview: {preview if preview else '[OCR unavailable or no confident text extracted]'}")
    print(f"  Parsed key-value pairs: {len(fields)}")
    for k, v in list(fields.items())[:8]:
        print(f"    - {k}: {v}")

print("\n" + "=" * 72)

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/dellpromax/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.



STAGE 4 (REAL DATA): OCR & FIELD EXTRACTION
Initializing PaddleOCR...
[WARN] OCR extractor init failed, continuing without OCR text: Engine 'paddle_static' is unavailable because dependency 'paddlepaddle' is not installed.

File: CoO UN template.png
  OCR text preview: [OCR unavailable or no confident text extracted]
  Parsed key-value pairs: 0

File: CoO WCO (kyoto).png
  OCR text preview: [OCR unavailable or no confident text extracted]
  Parsed key-value pairs: 0

File: certificat of origine..pdf
  OCR text preview: [OCR unavailable or no confident text extracted]
  Parsed key-value pairs: 0

File: certificat of origine.pdf
  OCR text preview: [OCR unavailable or no confident text extracted]
  Parsed key-value pairs: 0



---

# STAGE 5: Deduplication with Hash-Based IDs

Generate deterministic IDs and detect duplicate templates using content hashing.

In [ ]:
class TemplateDeduplicator:
    """
    Generate deterministic IDs and detect duplicates using SHA-1 hashing.
    """
    
    @staticmethod
    def generate_hash_id(
        template_name: str,
        page_number: int,
        descriptor: str,
        fields: Dict[str, str] = None,
    ) -> str:
        """
        Generate a deterministic hash-based ID.
        
        Args:
            template_name: Name of the template file
            page_number: Page index (1-based)
            descriptor: Structural descriptor text
            fields: Optional field dictionary for content signature
        
        Returns:
            SHA-1 hash ID (short form)
        """
        # Canonical signature: deterministic ordering
        canonical_sig = f"{template_name}#{page_number}#{descriptor}"
        
        if fields:
            # Add canonical field representation
            fields_str = '|'.join(
                f"{k}={v}" for k, v in sorted(fields.items())
            )
            canonical_sig += f"#{fields_str}"
        
        # Generate SHA-1 hash
        hash_obj = hashlib.sha1(canonical_sig.encode('utf-8'))
        short_hash = hash_obj.hexdigest()[:12]
        
        return f"tpl_{short_hash}"
    
    @staticmethod
    def detect_duplicates(templates: List[Dict]) -> Dict[str, List[Dict]]:
        """
        Group templates by their hash IDs to detect duplicates.
        
        Returns:
            Dict mapping hash_id → list of template records
        """
        groups = {}
        for template in templates:
            hash_id = template['hash_id']
            if hash_id not in groups:
                groups[hash_id] = []
            groups[hash_id].append(template)
        return groups


print("✓ Template Deduplicator class defined\n")

In [26]:
print("\n" + "=" * 72)
print("STAGE 5 (REAL DATA): DEDUPLICATION WITH HASH-BASED IDs")
print("=" * 72)

if "TemplateDeduplicator" not in globals():
    class TemplateDeduplicator:
        @staticmethod
        def generate_hash_id(
            template_name: str,
            page_number: int,
            descriptor: str,
            fields: Dict[str, str] | None = None,
        ) -> str:
            canonical_sig = f"{template_name}#{page_number}#{descriptor}"
            if fields:
                fields_str = "|".join(f"{k}={v}" for k, v in sorted(fields.items()))
                canonical_sig += f"#{fields_str}"
            return f"tpl_{hashlib.sha1(canonical_sig.encode('utf-8')).hexdigest()[:12]}"

        @staticmethod
        def detect_duplicates(templates: List[Dict]) -> Dict[str, List[Dict]]:
            groups: Dict[str, List[Dict]] = {}
            for template in templates:
                groups.setdefault(template["hash_id"], []).append(template)
            return groups

dedup = TemplateDeduplicator()
real_templates = []

for rec in template_records:
    if "ocr_text" not in rec:
        rec["ocr_text"] = ""

    fields = {}
    for line in rec["ocr_text"].splitlines():
        l = line.strip()
        if ":" in l:
            k, v = l.split(":", 1)
            fields[k.strip().lower()] = v.strip()

    hash_id = dedup.generate_hash_id(
        template_name=rec["template_file"],
        page_number=rec["page"],
        descriptor=rec["descriptor"],
        fields=fields if fields else None,
    )

    real_templates.append({
        "name": rec["template_file"],
        "page": rec["page"],
        "descriptor": rec["descriptor"],
        "fields": fields,
        "hash_id": hash_id,
    })

duplicate_groups = dedup.detect_duplicates(real_templates)

dup_sets = 0
unique_sets = 0
for hash_id, group in duplicate_groups.items():
    if len(group) > 1:
        dup_sets += 1
        print(f"\n[DUPLICATE SET] {hash_id}")
        for idx, item in enumerate(group, start=1):
            print(f"  Copy {idx}: {item['name']} (page {item['page']})")
    else:
        unique_sets += 1

print("\nSummary:")
print(f"  Total template pages processed : {len(real_templates)}")
print(f"  Unique signature groups        : {unique_sets}")
print(f"  Duplicate signature groups     : {dup_sets}")
print("\n" + "=" * 72)


STAGE 5 (REAL DATA): DEDUPLICATION WITH HASH-BASED IDs

Summary:
  Total template pages processed : 12
  Unique signature groups        : 12
  Duplicate signature groups     : 0



---

# SUMMARY: Complete Pipeline Visualization

This notebook implemented 5 progressive stages for robust template matching.

In [27]:
print("\n" + "#" * 72)
print("#" + " " * 70 + "#")
print("#" + "  ADVANCED TEMPLATE MATCHING PIPELINE - REAL DATA EXECUTION".center(70) + "#")
print("#" + " " * 70 + "#")
print("#" * 72)

print("\nRESULTS SUMMARY (REAL DOCUMENTS):")
print(f"  Stage 1 - Pixel similarity comparisons: {len(results_stage1) if 'results_stage1' in globals() else 0}")
if 'results_stage1' in globals() and results_stage1:
    print(f"    Example best match: {results_stage1[0]['query']} -> {results_stage1[0]['template']} (hybrid={results_stage1[0]['hybrid_score']:.4f})")

print(f"\n  Stage 2 - Embedding models compared: {len(embedding_results) if 'embedding_results' in globals() else 0}")
if 'embedding_results' in globals():
    for model_name in embedding_results.keys():
        print(f"    - {model_name}")

print(f"\n  Stage 3 - Two-stage retrieval queries processed: {len(matches) if 'matches' in globals() else 0}")
if 'matches' in globals() and matches:
    print(f"    Example top result: {matches[0]['query']} -> {matches[0]['template_name']} (score={matches[0]['hybrid_score']:.4f})")

print(f"\n  Stage 4 - OCR sample files processed: {len(real_ocr_samples) if 'real_ocr_samples' in globals() else 0}")
if 'real_ocr_samples' in globals() and real_ocr_samples:
    print(f"    First OCR sample: {real_ocr_samples[0]['file']}")

if 'duplicate_groups' in globals():
    dup_sets = sum(1 for g in duplicate_groups.values() if len(g) > 1)
    print(f"\n  Stage 5 - Dedup signature groups: {len(duplicate_groups)} total, {dup_sets} duplicate groups")

print("\n" + "=" * 72)
print("Notebook now runs on original data in templates/ and test/.")
print("If needed, increase OCR scope from first page to all pages per document.")
print("=" * 72 + "\n")


########################################################################
#                                                                      #
#       ADVANCED TEMPLATE MATCHING PIPELINE - REAL DATA EXECUTION      #
#                                                                      #
########################################################################

RESULTS SUMMARY (REAL DOCUMENTS):
  Stage 1 - Pixel similarity comparisons: 2
    Example best match: certificat of origine..pdf -> Egypt- CoO Templates.pdf (hybrid=0.3977)

  Stage 2 - Embedding models compared: 2
    - all-MiniLM-L6-v2
    - BAAI/bge-m3

  Stage 3 - Two-stage retrieval queries processed: 2
    Example top result: certificat of origine..pdf -> Egypt- CoO Templates.pdf (score=0.5996)

  Stage 4 - OCR sample files processed: 0

  Stage 5 - Dedup signature groups: 12 total, 0 duplicate groups

Notebook now runs on original data in templates/ and test/.
If needed, increase OCR scope from first page to all pages 